# Thống kê Quá trình Phân tích Cú pháp & Khử trùng lặp (Thực nghiệm 25.000 bài)

Notebook này nạp **số liệu thực nghiệm chính thức** của toàn bộ 5 thành viên (`TV1` – `TV5`, tổng cộng **25.000 bài báo**, **40.677 phiên bản**) từ tệp tổng hợp `summaries/milestone2/whole_team_summary.json` (được trích xuất trực tiếp từ `summaries/parsing_summary.json`, `summaries/parse-run.json`, `manifests/stages.json`, `acceptance-config.json` và `delivery.json` của từng thành viên, chụp trong khung thời gian `2026-11-20` – `2026-11-21` UTC).

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
summary_path = ROOT / 'summaries/milestone2/whole_team_summary.json'

if not summary_path.exists():
    raise FileNotFoundError(f"Không tìm thấy tệp tổng hợp MS2 tại: {summary_path}")

data = json.loads(summary_path.read_text(encoding='utf-8'))
agg = data['aggregate']
cov = agg['coverage_statistics']
dedup = agg['deduplication_statistics']
members = data['members']

display(Markdown(
    f"**Trạng thái dữ liệu:** `{data['data_status']}` | "
    f"**Khung snapshot (UTC):** `{data['snapshot_window_utc']}` | "
    f"**Môi trường chạy:** `{data['environment']['cpu_model']}` ({data['environment']['cpu_cores']} vCPU, {data['environment']['total_ram_gb']} GB RAM, Python {data['environment']['python_version']})"
))

## 1. Độ bao phủ & Trạng thái Hoàn thành (Cấp Bài báo và Phiên bản)

Trong hợp đồng Milestone 2, các bài báo hoàn tất và vượt qua kiểm định QA độc lập (`complete = true`) được chia tách rõ ràng giữa:
- **`success` (Parse TeX thành công):** có mã nguồn LaTeX, tạo đủ 3 bảng `elements`, `hierarchy`, `bibtex` và vượt kiểm định QA.
- **`no_source` (PDF-only hợp lệ):** không có mã nguồn TeX, được ghi nhận rỗng hợp lệ theo hợp đồng.
- **`incomplete` (Chưa hoàn chỉnh):** có ít nhất một phiên bản dừng ở `resolve`/`bibliography`, bị `validate_parse` từ chối, hoặc vượt timeout 300 giây.

In [ ]:
cov_rows = []
for m in members:
    mc = m['coverage_statistics']
    cov_rows.append({
        'Thành viên': m['member'],
        'Dải ID': m['assignment_range'],
        'Được giao': mc['total_assigned_publications'],
        'Parse TeX (success)': mc['successfully_parsed_publications'],
        'PDF-only (no_source)': mc['pdf_only_empty_publications'],
        'Đạt QA hoàn chỉnh': mc['certified_complete_publications'],
        'Tỷ lệ đạt QA (%)': f"{mc['certified_complete_rate_pct']:.2f}%",
        'Chưa hoàn chỉnh': mc['incomplete_publications'],
        'Tổng phiên bản': mc['total_versions_discovered'],
        'Version parse OK': mc['parsed_versions_count'],
        'Version no_source': mc['empty_versions_count'],
        'Version skipped': mc['skipped_versions_count'],
    })

cov_rows.append({
    'Thành viên': 'Toàn nhóm',
    'Dải ID': '5 dải (25.000 ID)',
    'Được giao': cov['total_assigned_publications'],
    'Parse TeX (success)': cov['successfully_parsed_publications'],
    'PDF-only (no_source)': cov['pdf_only_empty_publications'],
    'Đạt QA hoàn chỉnh': cov['certified_complete_publications'],
    'Tỷ lệ đạt QA (%)': f"{cov['certified_complete_rate_pct']:.2f}%",
    'Chưa hoàn chỉnh': cov['incomplete_publications'],
    'Tổng phiên bản': cov['total_versions_discovered'],
    'Version parse OK': cov['parsed_versions_count'],
    'Version no_source': cov['empty_versions_count'],
    'Version skipped': cov['skipped_versions_count'],
})
cov_df = pd.DataFrame(cov_rows)
display(cov_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8))

# 1. Biểu đồ cột chồng theo thành viên
labels = [m['member'] for m in members]
s_vals = np.array([m['coverage_statistics']['successfully_parsed_publications'] for m in members])
n_vals = np.array([m['coverage_statistics']['pdf_only_empty_publications'] for m in members])
i_vals = np.array([m['coverage_statistics']['incomplete_publications'] for m in members])

ax1.bar(labels, s_vals, label='Parse TeX (success)', color='#2ca02c', width=0.58)
ax1.bar(labels, n_vals, bottom=s_vals, label='PDF-only (no_source)', color='#1f77b4', width=0.58)
ax1.bar(labels, i_vals, bottom=s_vals + n_vals, label='Chưa hoàn chỉnh (incomplete)', color='#d62728', alpha=0.85, width=0.58)
ax1.set_ylabel('Số lượng bài báo')
ax1.set_ylim(0, 5800)
ax1.set_title('Phân bố Trạng thái 5.000 Bài báo theo Thành viên', fontweight='bold')
ax1.legend(loc='upper right', frameon=True, fontsize=9)
ax1.grid(axis='y', linestyle='--', alpha=0.6)
for idx in range(len(labels)):
    cert = s_vals[idx] + n_vals[idx]
    ax1.text(idx, cert / 2, f"{cert:,}\n({cert/50:.1f}%)", ha='center', va='center', color='white', fontweight='bold', fontsize=8.5)
    ax1.text(idx, cert + i_vals[idx] / 2, f"{i_vals[idx]:,}", ha='center', va='center', color='white', fontweight='bold', fontsize=8.5)

# 2. Biểu đồ cột cấp phiên bản toàn nhóm
ver_labels = ['Tổng khám phá', 'Parse thành công', 'PDF-only (no_source)', 'Bị bỏ qua (skipped)']
ver_counts = [
    cov['total_versions_discovered'],
    cov['parsed_versions_count'],
    cov['empty_versions_count'],
    cov['skipped_versions_count'],
]
ver_colors = ['#4c72b0', '#2ca02c', '#1f77b4', '#d62728']
bars = ax2.bar(ver_labels, ver_counts, color=ver_colors, width=0.55)
ax2.set_ylabel('Số lượng phiên bản')
ax2.set_title('Độ bao phủ Cấp Phiên bản Toàn nhóm (40.677 phiên bản)', fontweight='bold')
ax2.tick_params(axis='x', rotation=12)
ax2.grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars:
    yval = bar.get_height()
    pct = yval / cov['total_versions_discovered'] * 100
    ax2.text(bar.get_x() + bar.get_width() / 2, yval + 600, f"{yval:,}\n({pct:.1f}%)", ha='center', va='bottom', fontweight='bold', fontsize=8.5)
ax2.set_ylim(0, max(ver_counts) * 1.18)

plt.tight_layout()
plt.show()

## 2. Hiệu quả Khử trùng lặp Xuyên phiên bản (Cross-version Deduplication)

Thống kê dưới đây được tổng hợp trên **16.321 bài báo có mã nguồn TeX đạt chứng nhận hoàn chỉnh** (`complete = true`). Các dải bài cũ hơn (TV4, TV5) có số phiên bản trung bình trên mỗi bài cao hơn (1,66 – 1,67 phiên bản/bài) so với dải bài mới hơn (TV1: 1,57 phiên bản/bài), kéo theo tỷ lệ nén khử trùng lặp tăng tương ứng từ **30,93%** lên **35,13%** đối với `elements` và từ **36,35%** lên **40,63%** đối với `bibtex`.

In [ ]:
dedup_rows = []
for m in members:
    mc = m['coverage_statistics']
    md = m['deduplication_statistics']
    avg_ver = mc['total_versions_discovered'] / mc['total_assigned_publications']
    dedup_rows.append({
        'Thành viên': m['member'],
        'Version TB / bài': f"{avg_ver:.2f}",
        'Elements trước': f"{md['elements_before_dedup']:,}",
        'Elements sau': f"{md['elements_after_dedup']:,}",
        'Giảm Elements (%)': f"{md['element_dedup_reduction_pct']:.2f}%",
        'BibTeX trước': f"{md['bibtex_entries_before_dedup']:,}",
        'BibTeX sau': f"{md['bibtex_entries_after_dedup']:,}",
        'Giảm BibTeX (%)': f"{md['bibtex_dedup_reduction_pct']:.2f}%",
        'Lượt đổi tên \\cite{}': f"{md['citations_renamed_count']:,}",
    })

dedup_rows.append({
    'Thành viên': 'Toàn nhóm',
    'Version TB / bài': f"{cov['total_versions_discovered'] / cov['total_assigned_publications']:.2f}",
    'Elements trước': f"{dedup['elements_before_dedup']:,}",
    'Elements sau': f"{dedup['elements_after_dedup']:,}",
    'Giảm Elements (%)': f"{dedup['element_dedup_reduction_pct']:.2f}%",
    'BibTeX trước': f"{dedup['bibtex_entries_before_dedup']:,}",
    'BibTeX sau': f"{dedup['bibtex_entries_after_dedup']:,}",
    'Giảm BibTeX (%)': f"{dedup['bibtex_dedup_reduction_pct']:.2f}%",
    'Lượt đổi tên \\cite{}': f"{dedup['citations_renamed_count']:,}",
})
dedup_df = pd.DataFrame(dedup_rows)
display(dedup_df)

display(Markdown(
    f"**Tổng kết toàn nhóm:** Loại bỏ được **`{dedup['elements_removed']:,}`** phần tử nội dung trùng lặp "
    f"(**`{dedup['element_dedup_reduction_pct']:.2f}%`**), **`{dedup['bibtex_entries_removed']:,}`** mục BibTeX trùng lặp "
    f"(**`{dedup['bibtex_dedup_reduction_pct']:.2f}%`**), và đồng bộ đổi tên **`{dedup['citations_renamed_count']:,}`** lượt gọi `\\cite{{}}`."
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))

x = np.arange(len(members))
width = 0.36
el_before_m = [m['deduplication_statistics']['elements_before_dedup'] / 1e6 for m in members]
el_after_m = [m['deduplication_statistics']['elements_after_dedup'] / 1e6 for m in members]

r1 = ax1.bar(x - width/2, el_before_m, width, label='Trước Dedup', color='#d62728', alpha=0.85)
r2 = ax1.bar(x + width/2, el_after_m, width, label='Sau Dedup', color='#2ca02c', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels([m['member'] for m in members], fontweight='bold')
ax1.set_ylabel('Số phần tử (triệu)')
ax1.set_title('Số lượng Elements Trước và Sau Khử trùng lặp', fontweight='bold')
ax1.legend(frameon=True)
ax1.grid(axis='y', linestyle='--', alpha=0.6)
for rect in list(r1) + list(r2):
    h = rect.get_height()
    ax1.text(rect.get_x() + rect.get_width()/2, h + 0.04, f"{h:.2f}M", ha='center', va='bottom', fontsize=8, fontweight='bold')
ax1.set_ylim(0, max(el_before_m) * 1.18)

el_pct = [m['deduplication_statistics']['element_dedup_reduction_pct'] for m in members]
bib_pct = [m['deduplication_statistics']['bibtex_dedup_reduction_pct'] for m in members]
r3 = ax2.bar(x - width/2, el_pct, width, label='Giảm Elements (%)', color='#3b528b', alpha=0.88)
r4 = ax2.bar(x + width/2, bib_pct, width, label='Giảm BibTeX (%)', color='#ff7f0e', alpha=0.88)
ax2.set_xticks(x)
ax2.set_xticklabels([m['member'] for m in members], fontweight='bold')
ax2.set_ylabel('Tỷ lệ nén (%)')
ax2.set_title('Tỷ lệ Nén Khử trùng lặp theo Dải Thành viên (TV1 mới nhất → TV5 cũ hơn)', fontweight='bold')
ax2.legend(frameon=True)
ax2.grid(axis='y', linestyle='--', alpha=0.6)
for rect in list(r3) + list(r4):
    h = rect.get_height()
    ax2.text(rect.get_x() + rect.get_width()/2, h + 0.6, f"{h:.1f}%", ha='center', va='bottom', fontsize=8, fontweight='bold')
ax2.set_ylim(0, max(bib_pct) * 1.18)

plt.tight_layout()
plt.show()

## 3. Phễu Thực thi 6 Giai đoạn & Phân rã Nguyên nhân Lỗi Thực nghiệm

Tổng hợp trực tiếp từ `manifests/stages.json` của cả 5 thành viên cho thấy hai kết quả nổi bật:
1. **Trạm 2 (`hierarchy`) và Trạm 3 (`normalize`) đạt 100% thành công (`30.313 / 30.313` phiên bản, 0 lỗi)** trên toàn bộ các phiên bản vượt qua `resolve`; tương tự **Trạm 4B (`deduplicate`) đạt 100% thành công (`19.742 / 19.742` bài, 0 lỗi)**.
2. Toàn bộ các ca chưa hoàn chỉnh tập trung rõ rệt ở **Trạm 1 (`resolve`: 7.482 `data_error` + 240 `skipped` từ MS1)**, **Trạm 4A (`bibliography`: 411 `data_error`)**, và **Trạm 5 (`validate_parse`: 1.501 `data_error`, 100% do sai lệch round-trip `references.bib` ↔ `bibtex.json`)** cùng **1 bài timeout 300s (`2506.19089`)**.

In [ ]:
stage_meta = [
    ('resolve', 'Cấp phiên bản (TV1)'),
    ('hierarchy', 'Cấp phiên bản (TV2)'),
    ('normalize', 'Cấp phiên bản (TV3)'),
    ('bibliography', 'Cấp phiên bản (TV4)'),
    ('deduplicate', 'Cấp bài báo (TV4)*'),
    ('validate_parse', 'Cấp bài báo (TV5)*'),
]
stage_rows = []
for st, scope in stage_meta:
    d = data['stage_breakdown'][st]
    total = sum(d.values())
    stage_rows.append({
        'Giai đoạn': st,
        'Phạm vi': scope,
        'success': d.get('success', 0),
        'no_source': d.get('no_source', 0),
        'data_error': d.get('data_error', 0),
        'skipped': d.get('skipped', 0),
        'Tổng bản ghi': total,
    })
stage_df = pd.DataFrame(stage_rows)
display(stage_df)
display(Markdown("*Lưu ý:* Tổng số bản ghi ở `deduplicate` và `validate_parse` là `24.999` do 1 bài báo (`2506.19089` thuộc TV2) bị ngắt bởi giới hạn timeout 300 giây trước khi tới Trạm 4B."))

# Biểu đồ phân rã nguyên nhân lỗi
res_err = data['error_taxonomy']['resolve_non_success']
val_err = data['error_taxonomy']['validate_parse_data_error']
bib_err = data['error_taxonomy']['bibliography_data_error']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))

# Left: Top 8 resolve errors
top_res = list(res_err.items())[:8]
res_labels = [k for k, _ in top_res][::-1]
res_vals = [v for _, v in top_res][::-1]
bars1 = ax1.barh(res_labels, res_vals, color='#c44e52', alpha=0.88)
ax1.set_xlabel('Số phiên bản')
ax1.set_title('Top 8 Nguyên nhân Dừng ở Trạm 1: resolve (Tổng 7.722 phiên bản)', fontweight='bold')
ax1.grid(axis='x', linestyle='--', alpha=0.6)
for b in bars1:
    w = b.get_width()
    ax1.text(w + 45, b.get_y() + b.get_height()/2, f"{w:,}", va='center', fontsize=8.5, fontweight='bold')
ax1.set_xlim(0, max(res_vals) * 1.16)

# Right: 100% of validate_parse rejections
val_items = list(val_err.items())
val_labels = [k for k, _ in val_items][::-1]
val_vals = [v for _, v in val_items][::-1]
bars2 = ax2.barh(val_labels, val_vals, color='#dd8452', alpha=0.9)
ax2.set_xlabel('Số bài báo')
ax2.set_title('Phân rã 100% Từ chối QA ở Trạm 5: validate_parse (Tổng 1.501 bài)', fontweight='bold')
ax2.grid(axis='x', linestyle='--', alpha=0.6)
for b in bars2:
    w = b.get_width()
    ax2.text(w + 10, b.get_y() + b.get_height()/2, f"{w:,}", va='center', fontsize=8.5, fontweight='bold')
ax2.set_xlim(0, max(val_vals) * 1.18)

plt.tight_layout()
plt.show()

bib_df = pd.DataFrame([{'Loại lỗi bibliography (Trạm 4A)': k, 'Số phiên bản': v} for k, v in bib_err.items()])
display(bib_df)

## 4. Phân phối Số Phiên bản & Tác động tới Tỷ lệ Hoàn chỉnh

Vì hợp đồng MS2 áp dụng nguyên tắc toàn vẹn nghiêm ngặt (*all-or-nothing* — mọi phiên bản của một bài báo đều phải hợp lệ mới chứng nhận `complete = true`), xác suất một bài báo bị ảnh hưởng bởi lỗi mã nguồn LaTeX ở ít nhất một phiên bản tăng lên theo số lượng phiên bản của bài đó.

In [ ]:
ver_dist = data['version_distribution']
ver_df = pd.DataFrame([
    {
        'Nhóm số phiên bản': row['version_bucket'],
        'Tổng số bài': row['total_papers'],
        'Parse TeX (success)': row['parsed_tex_papers'],
        'PDF-only (no_source)': row['pdf_only_papers'],
        'Đạt QA hoàn chỉnh': row['certified_complete_papers'],
        'Chưa hoàn chỉnh': row['incomplete_papers'],
        'Tỷ lệ đạt QA (%)': f"{row['certified_complete_rate_pct']:.2f}%",
    }
    for row in ver_dist
])
display(ver_df)

plot_rows = [r for r in ver_dist if r['version_bucket'] != '0 (timeout)']
buckets = [f"{r['version_bucket']} ver" for r in plot_rows]
totals = [r['total_papers'] for r in plot_rows]
rates = [r['certified_complete_rate_pct'] for r in plot_rows]

fig, ax1 = plt.subplots(figsize=(9.5, 4.4))
bars = ax1.bar(buckets, totals, color='#6baed6', alpha=0.85, width=0.52, label='Tổng số bài báo')
ax1.set_ylabel('Số lượng bài báo', color='#08519c', fontweight='bold')
ax1.set_xlabel('Số phiên bản của bài báo')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars:
    h = b.get_height()
    ax1.text(b.get_x() + b.get_width()/2, h + 220, f"{h:,}", ha='center', va='bottom', fontsize=8.5, color='#08519c', fontweight='bold')
ax1.set_ylim(0, max(totals) * 1.15)

ax2 = ax1.twinx()
ax2.plot(buckets, rates, color='#d62728', marker='o', linewidth=2.3, markersize=7, label='Tỷ lệ đạt QA hoàn chỉnh (%)')
ax2.set_ylabel('Tỷ lệ đạt QA hoàn chỉnh (%)', color='#d62728', fontweight='bold')
ax2.set_ylim(45, 88)
for idx, r in enumerate(rates):
    ax2.text(idx, r + 1.8, f"{r:.1f}%", ha='center', va='bottom', fontsize=8.5, color='#d62728', fontweight='bold')

ax1.set_title('Phân phối Số Phiên bản và Xu hướng Tỷ lệ Hoàn chỉnh trên 24.999 Bài báo', fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', frameon=True)

plt.tight_layout()
plt.show()